# 04_2d_inversion_results

Inspect one multi-scale 2D inversion ladder. Scale is which inverted model (frequency of inversion plus iterate): compare inverted models with the true earth and slice resistivity. Data view is every frequency x source x receiver: synthetics versus observed gathers follow that view, which may be a different frequency from the scale. Generate synthetics from a selected iterate predicts every frequency (the earth model is not a single-tone gather). Export a chosen iterate to SEG-Y.

Run with Voila (`--strip_sources=True`).

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import traceback

# Project root: start scripts run from repo root, so cwd is the workshop directory
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.workshop_config import load_config
CONFIG = load_config()
WORKSPACE = CONFIG.workspace
INV_2D_RUNS_DIR = CONFIG.inv_2d_runs_dir
RESULTS_2D_DIR = CONFIG.results_2d_dir

import numpy as np
try:
    import ipywidgets as ipw
    import plotly.graph_objects as go
except Exception as exc:
    raise RuntimeError('Missing GUI dependencies. Install with: pip install voila ipywidgets plotly numpy ipykernel matplotlib scipy segyio') from exc

from scripts.modules import rockem_bridge
from scripts.modules.fd_visualization import (
    load_rss_traces,
    compute_gains_for_fd_outputs,
    apply_compare_plot_yaxes,
    view_combination_label,
    view_combinations,
)
from scripts.modules.inversion import (
    available_synthetic_pairs, find_observed_records,
)
from scripts.modules.multiscale_2d import (
    find_output_model, iter_stages, ladder_run_label, list_run_dirs,
    resample_model_log_rho, stage_label,
)
from scripts.modules.segy import write_resistivity_to_segy_from_template
from scripts.modules.setup_defaults import (
    default_frequencies as _default_frequencies,
    default_f_min_hz as _default_f_min_hz_meta,
    default_n_periods_extract as _default_n_periods_extract,
)


INV_2D_INPUT_DIR = CONFIG.inv_2d_input_dir

# --- acquisition matrix -----------------------------------------------------
# Step 01 emits one dataset per (frequency, source component). This notebook
# inspects one LADDER (`Run{N}`) at a time. Scale is the inverted model
# (frequency of inversion plus iterate). View is frequency x source x
# receiver of the whole matrix and does not follow the scale.
# `_select_dataset` rebinds `DS` from the VIEW, so Data-tab paths and
# extraction metadata follow the gather, not the inverted model.
#
# ONE mechanism, shared by every notebook - see `headless.DatasetPaths`. Loose
# path constants rebound through a hand-maintained `global` list leave one name
# per artifact to forget, and a forgotten one stays pinned to the forward ROOT
# in silence until some handler opens it - which is what `chainsweep.py` looks
# for.
FWD_ROOT = CONFIG.fwd_2d_dir


def available_datasets():
    from scripts.modules.headless import iter_datasets
    try:
        return iter_datasets(FWD_ROOT)
    except Exception:
        return []


def _select_dataset(name=None):
    """Bind `DS` to one dataset of the matrix; returns its manifest entry."""
    global DS
    from scripts.modules.headless import select_dataset
    chosen, DS = select_dataset(FWD_ROOT, name)
    return chosen


# Bind DS at import, so nothing below can read an unbound name and no
# root-bound placeholder ever exists for a later edit to leave behind.
_select_dataset(None)
MAX_CPUS = max(2, os.cpu_count() or 2)
# See scripts.modules.rockem_bridge: validated checkout, explicit engine.
def _read_forward_setup_meta():
    if not DS.setup_meta.exists():
        return {}
    try:
        return json.loads(DS.setup_meta.read_text())
    except Exception:
        return {}


def _default_f_min_hz(freqs):
    meta = _read_forward_setup_meta()
    v = meta.get('f_min_hz')
    if v:
        return float(v)
    return float(min(freqs)) if len(freqs) else 2000.0


def _mean_ep_rss_value(ep_path):
    from third_party.rockseis.io.rsfile import rsfile

    e = rsfile()
    e.read(str(ep_path))
    data = np.asarray(e.data, dtype=float)
    if data.size == 0:
        raise ValueError(f'Empty ep model: {ep_path}')
    return float(np.nanmean(data))


def _copy_forward_ep(src_ep, dest_dir, expected_eps_r):
    """Copy one dataset's ep.rss into dest_dir and verify its permittivity."""
    dest_dir = Path(dest_dir)
    dst = dest_dir / 'ep.rss'
    src_ep = Path(src_ep)
    if not src_ep.exists():
        raise FileNotFoundError(f'Missing forward ep model: {src_ep}')
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_ep, dst)
    expected = float(expected_eps_r)
    actual = _mean_ep_rss_value(dst)
    if not np.isclose(actual, expected, rtol=0.0, atol=1e-3):
        raise ValueError(
            f'Forward ep.rss permittivity ({actual:.4g}) does not match setup eps_r_used ({expected:.4g}). '
            'Re-run Step 01 finalize setup.'
        )
    return expected

VOILA_PID_FILES = [
    ROOT / '.voila_2d_results_server.pid',
]

state = {
    'last_messages': [],
    'initial_load': True,
    'fd_process': None,
    'real_result': None,
    'syn_result': None,
    'current_run_dir': None,
    'current_model_path': None,
    'current_x': None,
    'current_z': None,
    'updating_run': False,
    'updating_scale': False,
    'fd_running': False,
    'fd_stop': False,
}

def push_message(msg):
    state['last_messages'].append(msg)
    if len(state['last_messages']) > 12:
        state['last_messages'] = state['last_messages'][-12:]
    if 'status_out' in dir() and status_out is not None:
        status_out.value = '\n'.join(state['last_messages'])

def build_model_list(run_dir):
    """Iterates of one SCALE directory. The true model is the right-hand panel."""
    if run_dir is None:
        return []
    run_dir = Path(run_dir)
    options = []
    if (run_dir / "sg0.rss").exists():
        options.append(("sg0", run_dir / "sg0.rss"))
    if (run_dir / "sg_ls.rss").exists():
        options.append(("sg_ls", run_dir / "sg_ls.rss"))
    def _num(p):
        m = re.search(r"(\\d+)$", p.name)
        return int(m.group(1)) if m else 0
    for p in sorted(run_dir.glob("Results/sg_up.rss-*"), key=_num):
        options.append((p.name, p))
    return options


def get_real_data_paths(run_dir, source_field=None):
    """The observed Hx/Hz pair of a run, for ONE source component.

    A single-source run stages `Hx_data.rss`/`Hz_data.rss`; a JOINT run stages
    one file per (source, receiver), so which pair is meant depends on the
    source: `source_field='HX'` gives Cxx/Cxz and `'HZ'` gives Czx/Czz. That is
    the view selector's source axis, and it has to be passed in - defaulting to
    Kx would label the Kz components with the Kx data.
    `find_observed_records` is the one reader for both naming forms.
    """
    run_dir = Path(run_dir) if run_dir else None
    candidates = []
    if run_dir:
        candidates.append(run_dir)
        tag = run_dir / "dataset.txt"
        name = tag.read_text().strip() if tag.exists() else ""
        if name:
            candidates.append(INV_2D_INPUT_DIR / name)
    for d in candidates:
        obs = find_observed_records(d, source_field=source_field)
        if obs:
            return obs["HX"], obs["HZ"]
    missing = Path("Hx_data.rss")
    return missing, missing


def _read_rss_model(path):
    from third_party.rockseis.io.rsfile import rsfile
    f = rsfile()
    f.read(str(path))
    data = np.asarray(f.data, dtype=float)
    data = np.squeeze(data)
    if data.ndim != 2:
        raise ValueError(f'Expected 2D model for {path}, got shape {data.shape}')
    nx, nz = int(data.shape[0]), int(data.shape[1])
    grid = np.asarray(data.T, dtype=float)
    dx = float(f.geomD[0]) if f.geomD[0] else 1.0
    ox = float(f.geomO[0])
    iz = 2 if (len(f.geomN) > 2 and int(f.geomN[2]) > 0) else 1
    dz = float(f.geomD[iz]) if f.geomD[iz] else 1.0
    oz = float(f.geomO[iz])
    x = ox + dx * np.arange(nx)
    z = oz + dz * np.arange(nz)
    return x, z, grid

def _extract_positions(run_dir):
    run_dir = Path(run_dir)
    # Both staged naming forms: a single-source run's `Hx_data.rss` and a joint
    # run's `Hx_Hx_data.rss`. Every record of a run shares its geometry (the
    # engine refuses to start otherwise), so any of them gives the positions -
    # but hardcoding one name loses the TX/RX markers on a joint run entirely.
    hx_path = (find_observed_records(run_dir).get('HX')
               or find_observed_records(INV_2D_INPUT_DIR).get('HX'))
    if hx_path is None or not hx_path.exists():
        return np.array([]), np.array([]), np.array([]), np.array([])
    try:
        meta = load_rss_traces(hx_path)
        src = np.column_stack((meta['src_x'], meta['src_z']))
        rec = np.column_stack((meta['rx_x'], meta['rx_z']))
        src_u = np.unique(np.round(src, 6), axis=0)
        rec_u = np.unique(np.round(rec, 6), axis=0)
        return src_u[:, 0], src_u[:, 1], rec_u[:, 0], rec_u[:, 1]
    except Exception:
        return np.array([]), np.array([]), np.array([]), np.array([])

def _align_axes_to_survey(x, z, tx_x, tx_z, rx_x, rx_z):
    x, z = np.asarray(x, dtype=float), np.asarray(z, dtype=float)
    pts_x = np.concatenate([tx_x, rx_x]) if (tx_x.size + rx_x.size) > 0 else np.array([])
    pts_z = np.concatenate([tx_z, rx_z]) if (tx_z.size + rx_z.size) > 0 else np.array([])
    if pts_x.size == 0 or pts_z.size == 0:
        return x, z
    x_span = float(np.nanmax(x) - np.nanmin(x)) if x.size else 0.0
    z_span = float(np.nanmax(z) - np.nanmin(z)) if z.size else 0.0
    x_center = float(0.5 * (np.nanmin(x) + np.nanmax(x))) if x.size else 0.0
    z_center = float(0.5 * (np.nanmin(z) + np.nanmax(z))) if z.size else 0.0
    sx_center = float(0.5 * (np.nanmin(pts_x) + np.nanmax(pts_x)))
    sz_center = float(0.5 * (np.nanmin(pts_z) + np.nanmax(pts_z)))
    if x_span > 0.0 and abs(sx_center - x_center) > 0.5 * x_span:
        x = x + (sx_center - x_center)
    if z_span > 0.0 and abs(sz_center - z_center) > 0.5 * z_span:
        z = z + (sz_center - z_center)
    return x, z

def conductivity_to_resistivity(grid, min_sigma=1e-12):
    sigma = np.clip(np.asarray(grid, dtype=float), min_sigma, 1e12)
    return 1.0 / sigma

def resistivity_slice_at_x(x_arr, z_arr, grid, x_pick):
    x_arr = np.asarray(x_arr)
    idx = int(np.argmin(np.abs(x_arr - x_pick)))
    rho = conductivity_to_resistivity(grid[:, idx])
    return np.asarray(z_arr), np.asarray(rho)

def resistivity_slice_at_z(x_arr, z_arr, grid, z_pick):
    z_arr = np.asarray(z_arr)
    idx = int(np.argmin(np.abs(z_arr - z_pick)))
    rho = conductivity_to_resistivity(grid[idx, :])
    return np.asarray(x_arr), np.asarray(rho)

def infer_iteration_from_model_path(model_path, run_dir):
    model_path = Path(model_path)
    name = model_path.name
    m = re.search(r'sg_up\.rss-(\d+)$', name)
    if m:
        return int(m.group(1))
    if name == 'sg0.rss':
        return 0
    if name == 'sg_ls.rss':
        progress = Path(run_dir) / 'progress.log'
        if progress.exists():
            lines = progress.read_text(errors='replace').splitlines()
            iters = []
            for ln in lines:
                mm = re.match(r'^Iteration\s+(\d+)', ln.strip())
                if mm:
                    iters.append(int(mm.group(1)))
            if iters:
                return max(iters)
        return 0
    return 0


def find_synthetic_in_run(run_dir, iteration=None, source_field=None):
    """Modelled data for one source component, for the requested iteration.

    `source_field` picks which quarter of the tensor a joint run's four
    `data_mod_*` files to read, the same way `get_real_data_paths` does.

    Iteration handling: a request for a specific iteration is NOT satisfied by a
    different one - showing iteration 7's synthetics under a model labelled 3 is
    exactly the misleading case this guards. The engine's own un-suffixed
    `data_mod_*.rss` is a different matter: it is not another iteration, it is
    the modelled data of the LAST evaluation the run performed, carried at the
    negative sentinel indices. It is offered as a fallback and the caller says
    so, because otherwise a finished run - which writes no `-N` files at all -
    appears to have no synthetics.
    """
    pairs = available_synthetic_pairs(run_dir, source_field=source_field)
    if not pairs:
        return None, None, None

    if iteration is not None:
        for idx, hx, hz in pairs:
            if idx == int(iteration):
                return idx, hx, hz
        latest_engine = [p for p in pairs if p[0] < 0]
        if latest_engine:
            return latest_engine[-1]
        return None, None, None

    idx, hx, hz = pairs[-1]
    return idx, hx, hz


push_message('Results GUI loaded. Select a ladder, a scale, and an iterate.')


In [ ]:
from IPython.display import display, clear_output
from plotly.subplots import make_subplots

runs = list_run_dirs(INV_2D_RUNS_DIR)
run_options = [(ladder_run_label(p), str(p)) for _, p in runs] if runs else [('No runs', None)]

run_selector = ipw.Dropdown(
    options=run_options,
    value=run_options[0][1] if run_options and run_options[0][1] else None,
    description='Run:', layout=ipw.Layout(width='520px'),
)
scale_selector = ipw.Dropdown(
    options=[('No scales', None)], value=None,
    description='Scale:', layout=ipw.Layout(width='360px'),
)
model_selector = ipw.Dropdown(
    options=[], value=None, description='Iterate:', layout=ipw.Layout(width='400px'),
)
refresh_models_btn = ipw.Button(description='Refresh model list')
export_segy_btn = ipw.Button(description='Export selected model to SEGY')
export_status = ipw.HTML(value='')

# View is frequency x source x receiver of the whole acquisition matrix.
# Scale is the inverted model; the two are independent.
# Dropdown values are (dataset, source, receiver) tuples so rebuilding options
# cannot silently keep an integer that now names a different gather.
view_combo = ipw.Dropdown(
    options=[('(none built)', None)], value=None,
    description='view', layout=ipw.Layout(width='420px'),
    style={'description_width': '90px'},
)
dataset_info = ipw.HTML(value='No datasets found - run Step 01 first.')


def current_ladder_dir():
    v = run_selector.value
    return Path(v) if v else None


def current_stage():
    ladder = current_ladder_dir()
    if ladder is None:
        return None
    stages = iter_stages(ladder)
    want = scale_selector.value
    for s in stages:
        if str(s['path']) == str(want):
            return s
    return stages[-1] if stages else None


def selected_stage_dir():
    s = current_stage()
    return s['path'] if s else None


def _view_value(k):
    return (k['dataset'], k['source_field'], k['receiver'])


def current_view():
    """The selected `(frequency, source, receiver)` record, or None."""
    combos = state.get('view_combos') or []
    want = view_combo.value
    for k in combos:
        if _view_value(k) == want:
            return k
    return combos[0] if combos else None


def view_receiver():
    key = current_view()
    return key['receiver'] if key else 'Hx'


def view_source():
    """The selected SOURCE component (HX = Kx, HZ = Kz).

    On a joint run all four tensor components live in ONE run directory, so the
    source is what selects between Cxx/Cxz and Czx/Czz. Every reader of observed
    or modelled data has to be given it.
    """
    key = current_view()
    return key['source_field'] if key else 'HX'


def view_component_label():
    """`Cxx`/`Cxz`/`Czx`/`Czz` for the selected (source, receiver) pair."""
    return {('HX', 'Hx'): 'Cxx', ('HX', 'Hz'): 'Cxz',
            ('HZ', 'Hx'): 'Czx', ('HZ', 'Hz'): 'Czz'}.get(
                (view_source(), view_receiver()), view_receiver())


def scale_true_sg():
    """True sg.rss of the selected SCALE's forward group. Never DS.sg.

    Each frequency owns its own grid, so the Models tab must read the scale's
    Earth, not the Data view's dataset. Resolved by the scale's dataset.txt
    through group_datasets_by_frequency, then by frequency via ladder.json.
    """
    run_dir = selected_stage_dir()
    name = run_dataset_name(run_dir)
    from scripts.modules.headless import group_datasets_by_frequency
    if name:
        try:
            for grp in group_datasets_by_frequency(FWD_ROOT):
                if grp['name'] == name:
                    return Path(next(iter(grp['forward_dirs'].values()))) / 'sg.rss'
        except Exception:
            pass
        for entry in available_datasets():
            if entry.get('name') == name:
                return Path(entry['run_dir']) / 'sg.rss'
    stage = current_stage()
    if stage is not None and stage.get('freq_hz') is not None:
        f = float(stage['freq_hz'])
        try:
            for grp in group_datasets_by_frequency(FWD_ROOT):
                if grp.get('freq_hz') is not None and abs(float(grp['freq_hz']) - f) < 1e-3:
                    return Path(next(iter(grp['forward_dirs'].values()))) / 'sg.rss'
        except Exception:
            pass
        for entry in available_datasets():
            if entry.get('freq_hz') is not None and abs(float(entry['freq_hz']) - f) < 1e-3:
                return Path(entry['run_dir']) / 'sg.rss'
    return None


def view_observed_dir():
    """Observed-record directory for the VIEW frequency, not the scale.

    Prefer the ladder stage of this Run whose frequency matches the view, then
    inversion/input/<group> for that frequency, then the view dataset directory.
    """
    key = current_view()
    view_f = float(key['freq_hz']) if key and key.get('freq_hz') is not None else None
    src = view_source()
    ladder = current_ladder_dir()
    if ladder is not None and view_f is not None:
        for st in iter_stages(ladder):
            if st.get('freq_hz') is None:
                continue
            if abs(float(st['freq_hz']) - view_f) < 1e-3:
                p = Path(st['path'])
                if find_observed_records(p, source_field=src):
                    return p
                break
        from scripts.modules.headless import group_datasets_by_frequency
        try:
            for grp in group_datasets_by_frequency(FWD_ROOT):
                if grp.get('freq_hz') is not None and abs(float(grp['freq_hz']) - view_f) < 1e-3:
                    cand = INV_2D_INPUT_DIR / grp['name']
                    if find_observed_records(cand, source_field=src):
                        return cand
                    break
        except Exception:
            pass
    return DS.dir


def prediction_cache_dir(dataset_name=None):
    """`Run{N}/pred/<scaleName>_iterNNN/<dataset_name>/` for QC synthetics."""
    ladder = current_ladder_dir()
    stage = current_stage()
    if ladder is None or stage is None:
        return None
    run_dir = selected_stage_dir()
    iter_idx = infer_iteration_from_model_path(model_selector.value or '', run_dir)
    ds = dataset_name
    if ds is None:
        key = current_view()
        ds = key['dataset'] if key else None
    if not ds:
        return None
    scale_name = str(stage.get('name') or Path(stage['path']).name)
    return Path(ladder) / 'pred' / f'{scale_name}_iter{int(iter_idx):03d}' / ds


def refresh_view_combos():
    """Rebuild the view list for the whole matrix; keep the selected gather."""
    combos = view_combinations(available_datasets())
    prev = current_view()
    state['view_combos'] = [k for _lab, k in combos]
    options = ([(lab, _view_value(k)) for lab, k in combos]
               or [('(none built)', None)])
    view_combo.options = options
    chosen = None
    if prev is not None:
        want = _view_value(prev)
        values = [v for _, v in options]
        if want in values:
            chosen = want
        else:
            def _same_freq(a, b):
                fa, fb = a.get('freq_hz'), b.get('freq_hz')
                if fa is None and fb is None:
                    return True
                if fa is None or fb is None:
                    return False
                return abs(float(fa) - float(fb)) < 1e-3
            for k in state['view_combos']:
                if (_same_freq(k, prev)
                        and k['source_field'] == prev['source_field']
                        and k['receiver'] == prev['receiver']):
                    chosen = _view_value(k)
                    break
    view_combo.value = chosen if chosen is not None else options[0][1]
    # Always re-run the handler. Assigning `.value` fires NOTHING when the
    # value is unchanged, so the dropdown label and DS would diverge.
    on_dataset_change()


def on_dataset_change(_=None):
    key = current_view()
    if key is None:
        dataset_info.value = 'No datasets found - run Step 01 first.'
        return
    chosen = _select_dataset(key['dataset'])
    m = (chosen or {}).get('meta', {})
    lab = view_combination_label(key.get('freq_hz'), key['source_field'], key['receiver'])
    dataset_info.value = (
        f"<b>{lab}</b><br><code>{key['dataset']}</code>: source={m.get('source_field', '?')} "
        f"(source_type={m.get('source_type', '?')}), "
        f"flist={m.get('flist_hz')}, dx={m.get('dx_model_target_m')} m, "
        f"order={m.get('fd_order')} &mdash; showing receiver "
        f"<b>{key['receiver']}</b><br><small>{DS.dir}</small>"
    )
    push_message(f'View: {lab}')
    if state.get('real_result') is not None:
        on_load_real(None)
    if state.get('syn_result') is not None:
        on_load_syn_from_run(None)
    refresh_data_compare_controls()
    update_data_compare_plot()



# The observer is registered LATE, after every handler it calls exists - see
# the binding block near the end of this cell. Registering it here and then
# assigning `view_combo.value` would fire the callback into a half-built
# namespace.

model_plot_out = ipw.Output(layout=ipw.Layout(width='100%', height='500px'))
x_slice_input = ipw.FloatText(value=0.0, description='x (m):', layout=ipw.Layout(width='220px'))
z_slice_input = ipw.FloatText(value=0.0, description='z (m):', layout=ipw.Layout(width='220px'))
rho_z_out = ipw.Output(layout=ipw.Layout(width='100%', height='320px'))
rho_x_out = ipw.Output(layout=ipw.Layout(width='100%', height='320px'))

def setup_freqs():
    """The VIEW dataset's flist (the one tone in that gather)."""
    meta = _read_forward_setup_meta()
    vals = meta.get('flist_hz') or meta.get('freqs_hz')
    if vals:
        return np.asarray(vals, dtype=float)
    return np.asarray(_default_frequencies(path=DS.setup_meta), dtype=float)


load_real_btn = ipw.Button(description='Load real data')
load_syn_from_run_btn = ipw.Button(description='Load synthetics from run')
generate_syn_btn = ipw.Button(description='Generate synthetics from model', button_style='primary')
refresh_fd_btn = ipw.Button(description='Refresh FD status / load synthetics')
quit_btn = ipw.Button(description='Quit GUI server', button_style='danger', layout=ipw.Layout(width='170px'))
nproc_input = ipw.BoundedIntText(value=min(CONFIG.nproc_default, MAX_CPUS), min=2, max=MAX_CPUS, description='nproc', layout=ipw.Layout(width='200px'))
fd_status = ipw.HTML(value='')

# Data comparison controls (match visualize_gui modes)
metric_select = ipw.Dropdown(
    options=[
        ('Amplitude vs rx (local index)', 'amp_vs_rx'),
        ('Phase vs rx (deg, local index)', 'phase_vs_rx_deg'),
        ('Amplitude vs tx (fixed local rx)', 'amp_vs_tx'),
        ('Phase vs tx (deg, fixed local rx)', 'phase_vs_tx_deg'),
        ('Amplitude vs frequency', 'amp_vs_freq'),
        ('Phase vs frequency (deg)', 'phase_vs_freq_deg'),
    ],
    value='amp_vs_rx',
    description='plot',
)
tx_select = ipw.Dropdown(options=[('n/a', 0)], value=0, description='tx')
rx_local_select = ipw.IntSlider(value=0, min=0, max=0, step=1, description='local rx', continuous_update=False, layout=ipw.Layout(width='260px'))
trace_idx = ipw.IntSlider(value=0, min=0, max=0, step=1, description='trace idx', continuous_update=False, layout=ipw.Layout(width='320px'))

data_compare_out = ipw.Output(layout=ipw.Layout(width='100%', height='460px'))
status_out = ipw.Textarea(value='', description='Status', layout=ipw.Layout(width='100%', height='130px'))


def refresh_run_options():
    """Pick up ladder folders created since this GUI loaded."""
    runs = list_run_dirs(INV_2D_RUNS_DIR)
    options = [(ladder_run_label(p), str(p)) for _, p in runs] or [('No runs found', None)]
    current = run_selector.value
    state['updating_run'] = True
    run_selector.options = options
    values = [v for _, v in options]
    run_selector.value = current if current in values else values[0]
    state['updating_run'] = False


def refresh_scale_options(select_last=False):
    ladder = current_ladder_dir()
    state['updating_scale'] = True
    if ladder is None:
        scale_selector.options = [('No scales', None)]
        scale_selector.value = None
        state['updating_scale'] = False
        return
    stages = iter_stages(ladder)
    options = []
    prev = None
    for s in stages:
        options.append((stage_label(s, prev), str(s['path'])))
        prev = s
    if not options:
        options = [('No scales', None)]
    current = scale_selector.value
    scale_selector.options = options
    values = [v for _, v in options]
    if select_last:
        scale_selector.value = values[-1]
    elif current in values:
        scale_selector.value = current
    else:
        scale_selector.value = values[0]
    state['updating_scale'] = False


def refresh_model_options():
    refresh_run_options()
    run_dir = selected_stage_dir()
    if not run_dir:
        model_selector.options = [('Select inverted/linesearch model...', None)]
        model_selector.value = None
        return

    opts = build_model_list(Path(run_dir))

    # Keep startup view empty on inverted panel until user picks one.
    selector_options = [('Select inverted/linesearch model...', None)]
    selector_options.extend([(lbl, str(p)) for lbl, p in opts])
    model_selector.options = selector_options

    prev = state.get('current_model_path')
    valid_values = {val for _, val in selector_options}
    model_selector.value = prev if prev in valid_values else None

    state['current_run_dir'] = run_dir
    state['current_model_path'] = model_selector.value

def update_2d_plot():
    if state.get('initial_load'):
        return
    path = state.get('current_model_path')
    run_dir = state.get('current_run_dir')
    if not run_dir:
        with model_plot_out:
            clear_output(wait=True)
        return

    try:
        run_dir = Path(run_dir)
        tx_x, tx_z, rx_x, rx_z = _extract_positions(run_dir)

        true_sg = scale_true_sg()
        if true_sg is None or not Path(true_sg).exists():
            with model_plot_out:
                clear_output(wait=True)
                display(ipw.HTML('True model not found for the selected scale.'))
            return

        # True model defines fixed color limits and right panel reference.
        true_sg = Path(true_sg)
        x_true, z_true, g_true = _read_rss_model(true_sg)
        x_true, z_true = _align_axes_to_survey(x_true, z_true, tx_x, tx_z, rx_x, rx_z)
        rho_true = conductivity_to_resistivity(g_true)
        zmin = float(np.nanmin(rho_true))
        zmax = float(np.nanmax(rho_true))

        fig = make_subplots(rows=1, cols=2, subplot_titles=['Selected inverted/linesearch model', 'True model'], horizontal_spacing=0.10)

        selected_ready = False
        if path:
            p = Path(path)
            if p.exists() and str(p) != str(true_sg):
                x_sel, z_sel, g_sel = _read_rss_model(p)
                x_sel, z_sel = _align_axes_to_survey(x_sel, z_sel, tx_x, tx_z, rx_x, rx_z)
                rho_sel = conductivity_to_resistivity(g_sel)
                fig.add_trace(go.Heatmap(x=x_sel, y=z_sel, z=rho_sel, colorscale='Viridis', zmin=zmin, zmax=zmax, showscale=False), row=1, col=1)
                selected_ready = True
                state['current_x'] = float(np.mean(x_sel))
                state['current_z'] = float(np.mean(z_sel))

        if not selected_ready:
            fig.add_trace(go.Heatmap(x=x_true, y=z_true, z=np.full_like(rho_true, np.nan), colorscale='Viridis', zmin=zmin, zmax=zmax, showscale=False, hoverinfo='skip'), row=1, col=1)
            x_mid = float(np.mean(x_true)) if x_true.size else 0.0
            z_mid = float(np.mean(z_true)) if z_true.size else 0.0
            fig.add_annotation(
                x=x_mid,
                y=z_mid,
                text='Select an inverted/linesearch model from the list to display it here.',
                showarrow=False,
                xref='x',
                yref='y',
                font=dict(size=13),
                bgcolor='rgba(255,255,255,0.75)',
            )

        fig.add_trace(go.Heatmap(x=x_true, y=z_true, z=rho_true, colorscale='Viridis', zmin=zmin, zmax=zmax, showscale=True, colorbar=dict(title='Ohm.m', x=1.02, len=0.90)), row=1, col=2)

        for col, xx, zz in [(1, tx_x, tx_z), (2, tx_x, tx_z)]:
            if xx.size:
                fig.add_trace(go.Scatter(x=xx, y=zz, mode='markers', marker=dict(symbol='triangle-up', size=7, color='red'), name='TX', showlegend=(col == 1)), row=1, col=col)
        for col, xx, zz in [(1, rx_x, rx_z), (2, rx_x, rx_z)]:
            if xx.size:
                fig.add_trace(go.Scatter(x=xx, y=zz, mode='markers', marker=dict(symbol='circle', size=6, color='cyan'), name='RX', showlegend=(col == 1)), row=1, col=col)

        fig.update_xaxes(title_text='x (m)', row=1, col=1)
        fig.update_xaxes(title_text='x (m)', row=1, col=2, matches='x')
        fig.update_yaxes(title_text='z (m)', autorange='reversed', row=1, col=1)
        fig.update_yaxes(title_text='z (m)', autorange='reversed', row=1, col=2, matches='y')
        fig.update_layout(
            title='Resistivity models with survey overlay',
            height=520,
            margin=dict(t=80, b=60, l=40, r=110),
            legend=dict(orientation='h', y=-0.08),
            dragmode='zoom',
        )

        with model_plot_out:
            clear_output(wait=True)
            display(fig)

        if (not selected_ready) and (x_slice_input.value == 0.0 and z_slice_input.value == 0.0):
            state['current_x'] = float(np.mean(x_true)) if x_true.size else 0.0
            state['current_z'] = float(np.mean(z_true)) if z_true.size else 0.0

        if x_slice_input.value == 0.0 and z_slice_input.value == 0.0:
            x_slice_input.value = state.get('current_x', 0.0)
            z_slice_input.value = state.get('current_z', 0.0)
    except Exception as exc:
        push_message(f'2D plot error: {exc}')

def update_rho_plots():
    if state.get('initial_load'):
        return
    path = state.get('current_model_path')
    run_dir = state.get('current_run_dir')
    if not path or not run_dir:
        return
    try:
        path = Path(path)
        run_dir = Path(run_dir)
        x, z, grid = _read_rss_model(path)
        tx_x, tx_z, rx_x, rx_z = _extract_positions(run_dir)
        x, z = _align_axes_to_survey(x, z, tx_x, tx_z, rx_x, rx_z)
        x_pick = float(x_slice_input.value)
        z_pick = float(z_slice_input.value)

        z_ax, rho_z = resistivity_slice_at_x(x, z, grid, x_pick)
        x_ax, rho_x = resistivity_slice_at_z(x, z, grid, z_pick)

        z_curves = [np.asarray(rho_z, dtype=float)]
        x_curves = [np.asarray(rho_x, dtype=float)]

        fig_z = go.Figure()
        fig_z.add_trace(go.Scatter(x=rho_z, y=z_ax, mode='lines', name='Selected'))

        fig_x = go.Figure()
        fig_x.add_trace(go.Scatter(x=x_ax, y=rho_x, mode='lines', name='Selected'))

        true_sg = scale_true_sg()
        if true_sg is not None and Path(true_sg).exists() and str(path) != str(true_sg):
            xt, zt, gt = _read_rss_model(true_sg)
            xt, zt = _align_axes_to_survey(xt, zt, tx_x, tx_z, rx_x, rx_z)
            zt_ax, rho_tz = resistivity_slice_at_x(xt, zt, gt, x_pick)
            xt_ax, rho_tx = resistivity_slice_at_z(xt, zt, gt, z_pick)
            fig_z.add_trace(go.Scatter(x=rho_tz, y=zt_ax, mode='lines', name='True'))
            fig_x.add_trace(go.Scatter(x=xt_ax, y=rho_tx, mode='lines', name='True'))
            z_curves.append(np.asarray(rho_tz, dtype=float))
            x_curves.append(np.asarray(rho_tx, dtype=float))

        ladder = current_ladder_dir()
        if ladder is not None:
            for st in iter_stages(ladder):
                if str(st['path']) == str(run_dir):
                    continue
                mdl = find_output_model(st['path'])
                if mdl is None:
                    continue
                xs, zs, gs = _read_rss_model(mdl)
                xs, zs = _align_axes_to_survey(xs, zs, tx_x, tx_z, rx_x, rx_z)
                zs_ax, rho_os = resistivity_slice_at_x(xs, zs, gs, x_pick)
                fig_z.add_trace(go.Scatter(
                    x=rho_os, y=zs_ax, mode='lines',
                    name=f"{float(st['freq_hz']):g} Hz", line=dict(dash='dot')))


        fig_z.update_layout(
            title=f'Resistivity vs depth at x={x_pick:.1f} m',
            xaxis_title='Resistivity (Ohm.m)',
            yaxis_title='z (m)',
            yaxis_autorange='reversed',
            xaxis_autorange=True,
            yaxis_fixedrange=False,
            xaxis_fixedrange=False,
            height=300,
            margin=dict(t=50, b=40, l=50, r=20),
            legend=dict(orientation='h', y=-0.2),
        )
        with rho_z_out:
            clear_output(wait=True)
            display(fig_z)

        fig_x.update_layout(
            title=f'Resistivity vs x at z={z_pick:.1f} m',
            xaxis_title='x (m)',
            yaxis_title='Resistivity (Ohm.m)',
            yaxis_autorange=True,
            xaxis_autorange=True,
            xaxis_fixedrange=False,
            yaxis_fixedrange=False,
            height=300,
            margin=dict(t=50, b=40, l=50, r=20),
            legend=dict(orientation='h', y=-0.2),
        )
        with rho_x_out:
            clear_output(wait=True)
            display(fig_x)
    except Exception as exc:
        push_message(f'Slice plot error: {exc}')

def run_dataset_name(run_dir):
    """Forward group this SCALE inverted, from `<scale>/dataset.txt`."""
    if not run_dir:
        return None
    f = Path(run_dir) / 'dataset.txt'
    if not f.exists():
        return None
    name = f.read_text().strip()
    return name or None


def on_run_selected(change):
    if change.get('name') != 'value' or state.get('updating_run'):
        return
    refresh_scale_options(select_last=False)
    on_scale_selected({'name': 'value', 'new': scale_selector.value})


def on_scale_selected(change):
    if change.get('name') != 'value' or state.get('updating_scale'):
        return
    stage = current_stage()
    state['current_run_dir'] = str(stage['path']) if stage else None
    refresh_model_options()
    if not state.get('initial_load'):
        update_2d_plot()
        update_rho_plots()

def on_model_selected(change):
    if change.get('name') != 'value':
        return
    state['current_model_path'] = change.get('new')
    update_2d_plot()
    update_rho_plots()

def on_load_real(_):
    obs_dir = view_observed_dir()
    if not obs_dir:
        push_message('Select a view first.')
        return
    hx_path, hz_path = get_real_data_paths(Path(obs_dir), source_field=view_source())
    if not hx_path.exists() or not hz_path.exists():
        push_message('Real data files not found.')
        return
    try:
        freqs = setup_freqs()
        wav_path = Path(obs_dir) / 'wav2d.rss'
        if not wav_path.exists():
            wav_path = DS.wav2d
        result = compute_gains_for_fd_outputs(hx_path, hz_path, wav_path, freqs=freqs, f_min_hz=_default_f_min_hz(freqs), n_periods_extract=_default_n_periods_extract(path=DS.setup_meta))
        state['real_result'] = result
        refresh_data_compare_controls()
        key = current_view() or {}
        push_message(f'Real data loaded for {view_combination_label(key.get("freq_hz"), view_source(), view_receiver())}: {hx_path.name}, {hz_path.name}.')
        update_data_compare_plot()
    except Exception as exc:
        push_message(f'Load real failed: {exc}')

def _set_cfg_line(text, key, value):
    pat = re.compile(rf'^{re.escape(key)}\s*=.*?;$', re.MULTILINE)
    line = f'{key} = "{value}";'
    if pat.search(text):
        return pat.sub(line, text, count=1)
    return text + '\n' + line + '\n'


def _resolve_segy_template_path():
    if DS.setup_meta.exists():
        try:
            meta = json.loads(DS.setup_meta.read_text())
            p_raw = meta.get('segy_template_path')
            if p_raw:
                p = Path(p_raw).expanduser()
                if not p.is_absolute():
                    p = (ROOT / p).resolve()
                if p.exists():
                    return p, 'setup_metadata.json'
                push_message(f'SEG-Y template listed in setup metadata does not exist: {p}')
        except Exception as exc:
            push_message(f'Failed reading setup metadata for SEG-Y template: {exc}')

    fallback = CONFIG.default_segy
    if fallback.exists():
        return fallback, 'fallback default SEG-Y from config'
    return None, None


def _export_name_for_model(model_path, run_dir):
    model_path = Path(model_path)
    run_dir = Path(run_dir)
    name = model_path.name
    m = re.search(r'sg_up\.rss-(\d+)$', name)
    if m:
        return f'sg_up_iter{int(m.group(1)):03d}.segy', f'iteration {int(m.group(1))}'
    if name == 'sg0.rss':
        return 'sg0_iter000.segy', 'iteration 0 (initial model)'
    if name == 'sg_ls.rss':
        progress = run_dir / 'progress.log'
        if progress.exists():
            lines = progress.read_text(errors='replace').splitlines()
            iters = []
            for ln in lines:
                mm = re.match(r'^Iteration\s+(\d+)', ln.strip())
                if mm:
                    iters.append(int(mm.group(1)))
            if iters:
                it_last = max(iters)
                return f'sg_ls_iter{it_last:03d}.segy', f'line-search model near iteration {it_last}'
        return 'sg_ls_iterunknown.segy', 'line-search model (iteration unknown)'
    safe = model_path.stem.replace('.', '_')
    return f'{safe}.segy', f'model {model_path.name}'


def on_export_selected_segy(_):
    run_dir = selected_stage_dir()
    model_path = model_selector.value
    if not run_dir:
        push_message('Select a run and scale first.')
        return
    if not model_path or not Path(model_path).exists():
        push_message('Select an existing model first.')
        return

    run_dir = Path(run_dir)
    model_path = Path(model_path)
    export_status.value = '<span style="color:#333333">SEGY export in progress...</span>'
    template_path, template_src = _resolve_segy_template_path()
    if template_path is None:
        push_message('No SEG-Y template found. Run 01_fw_setup and finalize setup, or configure default SEG-Y in Step 00.')
        export_status.value = '<span style="color:#aa0000">SEGY export failed: no template SEG-Y found.</span>'
        return

    try:
        x, z, sigma = _read_rss_model(model_path)
        rho = conductivity_to_resistivity(sigma)
        ladder = current_ladder_dir()
        stage = current_stage()
        out_dir = RESULTS_2D_DIR / (ladder.name if ladder else 'Run') / run_dir.name
        out_name, model_desc = _export_name_for_model(model_path, run_dir)
        if stage is not None:
            out_name = f"{float(stage['freq_hz']):g}Hz_{out_name}"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / out_name
        info = write_resistivity_to_segy_from_template(
            template_segy_path=template_path,
            output_segy_path=out_path,
            resistivity_grid=rho,
            x_model=x,
            z_model=z,
            method='linear',
        )
        push_message(f'SEG-Y template: {template_path} ({template_src})')
        if info.get('interpolated'):
            push_message('Export grid differs from template; interpolated resistivity to exact SEG-Y template geometry.')
        else:
            push_message('Export grid already matches template geometry; wrote values without interpolation.')
        push_message('SEGY export SUCCESS.')
        push_message(f'Exported model: {model_desc}')
        push_message(f'Output file name: {out_path.name}')
        push_message(f'Output folder: {out_path.parent}')
        push_message('Export mode: overwrite enabled for same model/iteration filename.')
        export_status.value = (
            f'<span style="color:#006400">SEGY export succeeded. '
            f'File: {out_path.name} | Folder: {out_path.parent}</span>'
        )
    except Exception as exc:
        push_message(f'SEG-Y export failed: {exc}')
        export_status.value = f'<span style="color:#aa0000">SEGY export failed: {exc}</span>'


def on_load_syn_from_run(_):
    run_dir = selected_stage_dir()
    if not run_dir:
        push_message('Select a run and scale first.')
        fd_status.value = '<span style="color:#aa0000">Select a run first.</span>'
        return
    run_dir = Path(run_dir)
    desired_iter = infer_iteration_from_model_path(model_selector.value or '', run_dir)
    cache = prediction_cache_dir()
    search_dirs = []
    if cache is not None:
        search_dirs.append(Path(cache))
    key = current_view()
    stage = current_stage()
    if (stage is not None and key is not None and key.get('freq_hz') is not None
            and stage.get('freq_hz') is not None
            and abs(float(stage['freq_hz']) - float(key['freq_hz'])) < 1e-3):
        search_dirs.append(run_dir)
    fd_status.value = f'Loading synthetics (target iter={desired_iter})...'
    push_message(f'Searching synthetics for {view_combination_label((key or {}).get("freq_hz"), view_source(), view_receiver())} iter={desired_iter}.')
    push_message(f'Reading the {view_source()} source. Accepted patterns: '
                 'data_Hx_mod.rss-<iter>, data_mod_HX.rss-<iter>, the source-tagged '
                 'data_mod_<SRC>_HX.rss-<iter> a joint run writes, and the '
                 'un-suffixed forms of each.')

    idx, hx_path, hz_path = None, None, None
    used_dir = None
    for d in search_dirs:
        if d is None or not Path(d).exists():
            continue
        idx, hx_path, hz_path = find_synthetic_in_run(
            Path(d), iteration=desired_iter, source_field=view_source())
        if hx_path is not None and hz_path is not None:
            used_dir = Path(d)
            break
    if hx_path is None or hz_path is None:
        msg = f'No synthetic files found for selected model iteration {desired_iter}.'
        push_message(msg)
        fd_status.value = f'<span style="color:#aa0000">{msg}</span>'
        return
    try:
        freqs = setup_freqs()
        wav_path = used_dir / 'wav2d.rss' if used_dir is not None else DS.wav2d
        if not wav_path.exists():
            wav_path = DS.wav2d
        state['syn_result'] = compute_gains_for_fd_outputs(hx_path, hz_path, wav_path, freqs=freqs, f_min_hz=_default_f_min_hz(freqs), n_periods_extract=_default_n_periods_extract(path=DS.setup_meta))
        refresh_data_compare_controls()
        push_message(f'Loaded synthetic pair iter={idx} from run '
                     f'({view_source()} source).'
                     + (' Iteration -1/-2 means the engine\'s own un-suffixed '
                        'data_mod_* - the last evaluation, not iteration '
                        f'{desired_iter}.' if idx < 0 else ''))
        push_message(f'Synthetic files: {hx_path.name}, {hz_path.name}')
        fd_status.value = f'Synthetic loaded: {hx_path.name}, {hz_path.name}'
        if state.get('real_result') is None:
            push_message('Synthetic loaded; load real data to compare.')
        update_data_compare_plot()
    except Exception as exc:
        push_message(f'Load synthetics from run failed: {exc}')
        fd_status.value = f'<span style="color:#aa0000">Synthetic load failed: {exc}</span>'


def on_generate_synthetics(_):
    """Predict EVERY frequency from the selected iterate (resample onto each grid).

    Writes into `Run{N}/pred/<scale>_iterNNN/<dataset>/` so inversion-engine
    `data_mod_*` in other stages is not overwritten.
    """
    run_dir = selected_stage_dir()
    model_path = model_selector.value
    if not run_dir:
        push_message('Select a run and scale first.')
        return
    if not model_path or not Path(model_path).exists():
        push_message('Select an existing model first.')
        return
    if state.get('fd_running') or (state.get('fd_process') and state['fd_process'].poll() is None):
        push_message('Synthetic modelling already in progress.')
        return

    datasets = available_datasets()
    if not datasets:
        push_message('No forward datasets. Run Step 01 first.')
        return

    run_dir = Path(run_dir)
    model_path = Path(model_path)
    iter_idx = infer_iteration_from_model_path(model_path, run_dir)
    nproc = max(2, min(int(nproc_input.value), MAX_CPUS))
    state['fd_stop'] = False
    state['fd_running'] = True
    fd_status.value = f'Synthetic modelling: 0/{len(datasets)} datasets...'
    push_message(
        f'Generating synthetics for {len(datasets)} dataset(s) from iterate {iter_idx} '
        f'(resampled onto each frequency grid).')

    def _worker():
        from scripts.modules.headless import dataset_paths
        try:
            n = len(datasets)
            for i_ds, entry in enumerate(datasets, 1):
                if state.get('fd_stop'):
                    push_message('Synthetic modelling stopped.')
                    break
                name = entry['name']
                dest = prediction_cache_dir(dataset_name=name)
                if dest is None:
                    push_message(f'[{i_ds}/{n}] {name}: no prediction cache path.')
                    continue
                dest = Path(dest)
                dest.mkdir(parents=True, exist_ok=True)
                try:
                    dsp = dataset_paths(entry['run_dir'])
                    if not dsp.sg.exists():
                        raise FileNotFoundError(f'Missing template sg.rss: {dsp.sg}')
                    if not dsp.mod_cfg.exists():
                        raise FileNotFoundError(f'Missing FD template cfg: {dsp.mod_cfg}')
                    resample_model_log_rho(model_path, dsp.sg, dest / 'sg.rss')
                    meta = {}
                    if dsp.setup_meta.exists():
                        meta = json.loads(dsp.setup_meta.read_text())
                    if 'eps_r_used' not in meta:
                        raise ValueError(f'eps_r_used missing in {dsp.setup_meta}')
                    eps_r = _copy_forward_ep(dsp.ep, dest, meta['eps_r_used'])
                    for src_p, fname in ((dsp.wav2d, 'wav2d.rss'), (dsp.survey, 'Survey.rss')):
                        if not src_p.exists():
                            raise FileNotFoundError(f'Missing {fname}: {src_p}')
                        shutil.copy2(src_p, dest / fname)
                    hx_name = f'data_mod_HX.rss-{iter_idx}'
                    hz_name = f'data_mod_HZ.rss-{iter_idx}'
                    cfg_txt = dsp.mod_cfg.read_text()
                    cfg_txt = _set_cfg_line(cfg_txt, 'Sg', 'sg.rss')
                    cfg_txt = _set_cfg_line(cfg_txt, 'Ep', 'ep.rss')
                    cfg_txt = _set_cfg_line(cfg_txt, 'Wavelet', 'wav2d.rss')
                    cfg_txt = _set_cfg_line(cfg_txt, 'Survey', 'Survey.rss')
                    cfg_txt = _set_cfg_line(cfg_txt, 'Hxrecordfile', hx_name)
                    cfg_txt = _set_cfg_line(cfg_txt, 'Hzrecordfile', hz_name)
                    (dest / 'mod.cfg').write_text(cfg_txt)
                    engine = str(meta.get('forward_engine', CONFIG.forward_engine_te2d()))
                    bin_path = str(rockem_bridge.binary_path(engine))
                    fd_status.value = f'Synthetic modelling: {i_ds}/{n} {name}...'
                    push_message(
                        f'[{i_ds}/{n}] {name}: mpiEmmodTE2d into {dest.relative_to(current_ladder_dir())} '
                        f'(eps_r_used={eps_r:.4f}).')
                    proc = subprocess.Popen(
                        [CONFIG.mpirun, '-np', str(nproc), bin_path, 'mod.cfg'],
                        cwd=str(dest),
                    )
                    state['fd_process'] = proc
                    rc = proc.wait()
                    if rc != 0:
                        push_message(f'[{i_ds}/{n}] {name}: finished rc={rc}')
                    else:
                        push_message(f'[{i_ds}/{n}] {name}: finished rc=0')
                except Exception as exc:
                    push_message(f'[{i_ds}/{n}] {name}: FAILED {exc}')
            push_message('Synthetic modelling of the acquisition matrix finished.')
            fd_status.value = 'Synthetic modelling finished.'
            on_load_syn_from_run(None)
        finally:
            state['fd_process'] = None
            state['fd_running'] = False

    threading.Thread(target=_worker, daemon=True).start()


def on_refresh_fd_status(_):
    if state.get('fd_running'):
        proc = state.get('fd_process')
        if proc is not None and proc.poll() is None:
            fd_status.value = 'Synthetic modelling still running...'
        else:
            fd_status.value = 'Synthetic modelling running (between datasets)...'
        return
    on_load_syn_from_run(None)

def refresh_data_compare_controls():
    base = state.get('real_result') or state.get('syn_result')
    if base is None:
        tx_select.options = [('n/a', 0)]
        rx_local_select.min = 0
        rx_local_select.max = 0
        rx_local_select.value = 0
        trace_idx.max = 0
        trace_idx.value = 0
        return

    geo = base['geometry']
    comp_data = base.get(view_receiver(), base.get('Hx', {}))
    freqs = np.asarray(comp_data.get('freqs', []), dtype=float)
    if freqs.size == 0:
        push_message('No frequencies available in loaded data.')

    tx_vals = np.unique(np.asarray(geo.get('tx_idx_per_trace', []), dtype=int))
    if tx_vals.size:
        tx_select.options = [(f'Tx {int(v)}', int(v)) for v in tx_vals]
        tx_select.value = int(tx_vals[0])
    else:
        tx_select.options = [('n/a', 0)]
        tx_select.value = 0

    rx_local = np.asarray(geo.get('rx_local_idx_per_trace', geo.get('rx_idx_per_trace', [])), dtype=int)
    if rx_local.size:
        rx_local_select.min = int(np.nanmin(rx_local))
        rx_local_select.max = int(np.nanmax(rx_local))
        rx_local_select.value = int(rx_local_select.min)
    else:
        rx_local_select.min = 0
        rx_local_select.max = 0
        rx_local_select.value = 0

    ntr = int(np.asarray(geo.get('tx_idx_per_trace', [])).size)
    if ntr > 0:
        trace_idx.max = ntr - 1
        trace_idx.value = 0
    else:
        trace_idx.max = 0
        trace_idx.value = 0
        push_message('No traces available in loaded data.')


def _same_geometry(real, syn):
    if real is None or syn is None:
        return False
    rg = real['geometry']
    sg = syn['geometry']
    return (
        np.asarray(rg['tx_idx_per_trace']).shape == np.asarray(sg['tx_idx_per_trace']).shape
        and np.asarray(rg['rx_idx_per_trace']).shape == np.asarray(sg['rx_idx_per_trace']).shape
    )


def _phase_deg(arr):
    return np.rad2deg(np.asarray(arr, dtype=float))


def update_data_compare_plot(*_):
    real = state.get('real_result')
    syn = state.get('syn_result')

    with data_compare_out:
        clear_output(wait=True)
        if real is None and syn is None:
            return
        if real is None:
            display(ipw.HTML('Synthetic loaded. Load real data to compare.'))
            return

        key = current_view()
        comp = view_receiver()
        metric = metric_select.value
        # The tone comes from the same selector. A per-frequency dataset holds
        # exactly one, so this is index 0 there; a legacy broadband gather still
        # resolves to the nearest tone it actually carries.
        _f_all = np.asarray((real.get(comp) or {}).get('freqs', []), dtype=float)
        fidx = 0
        if key is not None and key.get('freq_hz') is not None and _f_all.size:
            fidx = int(np.argmin(np.abs(_f_all - float(key['freq_hz']))))
        tx_id = int(tx_select.value) if tx_select.value is not None else 0
        rx_local_target = int(rx_local_select.value)
        tr_pick = int(trace_idx.value)

        comp_data = real.get(comp)
        if comp_data is None:
            display(ipw.HTML(f'Missing {comp} data in real result.'))
            return

        geo = real['geometry']
        tx_arr = np.asarray(geo.get('tx_idx_per_trace', []), dtype=int)
        rx_arr = np.asarray(geo.get('rx_idx_per_trace', []), dtype=int)
        rx_local = np.asarray(geo.get('rx_local_idx_per_trace', rx_arr), dtype=int)
        freqs = np.asarray(comp_data.get('freqs', []), dtype=float)

        if tx_arr.size == 0 or rx_local.size == 0:
            display(ipw.HTML('Loaded data has no trace geometry to plot.'))
            return
        if freqs.size == 0:
            display(ipw.HTML('Loaded data has no frequencies to plot.'))
            return

        fidx = int(max(0, min(fidx, freqs.size - 1)))

        have_syn = syn is not None and _same_geometry(real, syn) and comp in syn
        syn_comp = syn.get(comp) if have_syn else None

        fig = go.Figure()
        title = ''

        if metric == 'amp_vs_rx':
            idx = np.where(tx_arr == tx_id)[0]
            if idx.size == 0:
                display(ipw.HTML('No traces for selected tx.'))
                return
            x = rx_local[idx]
            order = np.argsort(x)
            x = x[order]
            y_real = np.asarray(comp_data['amp_mean'][fidx, idx], dtype=float)[order]
            fig.add_trace(go.Scatter(x=x, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = np.asarray(syn_comp['amp_mean'][fidx, idx], dtype=float)[order]
                fig.add_trace(go.Scatter(x=x, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{view_component_label()} ({comp}) amplitude vs local rx (Tx {tx_id}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Local rx index')
            fig.update_yaxes(title_text='Amplitude')

        elif metric == 'phase_vs_rx_deg':
            idx = np.where(tx_arr == tx_id)[0]
            if idx.size == 0:
                display(ipw.HTML('No traces for selected tx.'))
                return
            x = rx_local[idx]
            order = np.argsort(x)
            x = x[order]
            y_real = _phase_deg(comp_data['phi_mean_rad'][fidx, idx])[order]
            fig.add_trace(go.Scatter(x=x, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = _phase_deg(syn_comp['phi_mean_rad'][fidx, idx])[order]
                fig.add_trace(go.Scatter(x=x, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{view_component_label()} ({comp}) phase vs local rx (Tx {tx_id}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Local rx index')
            fig.update_yaxes(title_text='Phase (deg)')

        elif metric == 'amp_vs_tx':
            tx_vals = np.unique(tx_arr)
            xs, y_real, y_syn = [], [], []
            for t in tx_vals:
                cand = np.where((tx_arr == int(t)) & (rx_local == rx_local_target))[0]
                if cand.size == 0:
                    continue
                i = int(cand[0])
                xs.append(int(t))
                y_real.append(float(comp_data['amp_mean'][fidx, i]))
                if syn_comp is not None:
                    y_syn.append(float(syn_comp['amp_mean'][fidx, i]))
            if not xs:
                display(ipw.HTML('No traces for selected local rx across tx.'))
                return
            fig.add_trace(go.Scatter(x=xs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                fig.add_trace(go.Scatter(x=xs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{view_component_label()} ({comp}) amplitude vs tx (local rx {rx_local_target}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Tx index')
            fig.update_yaxes(title_text='Amplitude')

        elif metric == 'phase_vs_tx_deg':
            tx_vals = np.unique(tx_arr)
            xs, y_real, y_syn = [], [], []
            for t in tx_vals:
                cand = np.where((tx_arr == int(t)) & (rx_local == rx_local_target))[0]
                if cand.size == 0:
                    continue
                i = int(cand[0])
                xs.append(int(t))
                y_real.append(float(_phase_deg(comp_data['phi_mean_rad'][fidx, i])))
                if syn_comp is not None:
                    y_syn.append(float(_phase_deg(syn_comp['phi_mean_rad'][fidx, i])))
            if not xs:
                display(ipw.HTML('No traces for selected local rx across tx.'))
                return
            fig.add_trace(go.Scatter(x=xs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                fig.add_trace(go.Scatter(x=xs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{view_component_label()} ({comp}) phase vs tx (local rx {rx_local_target}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Tx index')
            fig.update_yaxes(title_text='Phase (deg)')

        elif metric == 'amp_vs_freq':
            tr_pick = max(0, min(tr_pick, tx_arr.size - 1))
            y_real = np.asarray(comp_data['amp_mean'][:, tr_pick], dtype=float)
            fig.add_trace(go.Scatter(x=freqs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = np.asarray(syn_comp['amp_mean'][:, tr_pick], dtype=float)
                fig.add_trace(go.Scatter(x=freqs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{view_component_label()} ({comp}) amplitude vs frequency (trace {tr_pick}, tx={tx_arr[tr_pick]}, local_rx={rx_local[tr_pick]})'
            fig.update_xaxes(title_text='Frequency (Hz)', type='log')
            fig.update_yaxes(title_text='Amplitude')

        elif metric == 'phase_vs_freq_deg':
            tr_pick = max(0, min(tr_pick, tx_arr.size - 1))
            y_real = _phase_deg(comp_data['phi_mean_rad'][:, tr_pick])
            fig.add_trace(go.Scatter(x=freqs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = _phase_deg(syn_comp['phi_mean_rad'][:, tr_pick])
                fig.add_trace(go.Scatter(x=freqs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{view_component_label()} ({comp}) phase vs frequency (trace {tr_pick}, tx={tx_arr[tr_pick]}, local_rx={rx_local[tr_pick]})'
            fig.update_xaxes(title_text='Frequency (Hz)', type='log')
            fig.update_yaxes(title_text='Phase (deg)')

        else:
            display(ipw.HTML('Unknown plotting mode.'))
            return

        y_all = []
        for tr in fig.data:
            if getattr(tr, 'y', None) is not None:
                y_all.append(np.asarray(tr.y, dtype=float))
        y_vals = np.concatenate(y_all) if y_all else np.array([])
        y_vals = y_vals[np.isfinite(y_vals)] if y_vals.size else y_vals

        fig.update_layout(
            title=title,
            height=420,
            margin=dict(t=60, b=50, l=60, r=20),
            legend=dict(orientation='h', y=-0.2),
            xaxis_fixedrange=False,
            yaxis_fixedrange=False,
        )

        # Keep horizontal axis behavior unchanged (as before).
        fig.update_xaxes(autorange=True)

        apply_compare_plot_yaxes(fig, metric, y_vals)

        display(fig)

        if syn is not None and syn_comp is None:
            display(ipw.HTML('<span style="color:#b26a00">Synthetic geometry or component differs from real data; only real curve shown for this mode.</span>'))

def on_quit_gui(_):
    push_message('Shutting down results GUI server...')
    fd_status.value = 'Shutting down GUI server...'
    try:
        state['fd_stop'] = True
        proc = state.get('fd_process')
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except Exception:
                proc.kill()
                proc.wait(timeout=5)
    except Exception as exc:
        push_message(f'Quit warning while stopping FD process: {exc}')

    try:
        signaled = False
        for pid_file in VOILA_PID_FILES:
            if not pid_file.exists():
                continue
            pid_text = pid_file.read_text().strip()
            if not pid_text:
                continue
            pid = int(pid_text)
            if pid == os.getpid():
                continue

            try:
                os.kill(pid, signal.SIGTERM)
                push_message(f'Sent SIGTERM to Voila server PID {pid} ({pid_file.name}).')
                signaled = True
            except Exception as exc:
                push_message(f'Could not signal Voila PID {pid} from {pid_file.name}: {exc}')

        if not signaled:
            push_message('No external Voila PID found to terminate; shutting down kernel only.')
    except Exception as exc:
        push_message(f'Quit warning: {exc}')

    # Always request kernel shutdown, then force-exit as final fallback.
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and getattr(ip, 'kernel', None):
            ip.kernel.do_shutdown(restart=False)
    except Exception as exc:
        push_message(f'Kernel shutdown warning: {exc}')

    try:
        os.kill(os.getpid(), signal.SIGTERM)
    except Exception:
        pass


def bind_button_with_feedback(button, handler, action_label, success_message=None):
    def _wrapped(_):
        before = len(state.get('last_messages', []))
        push_message(f'{action_label}...')
        try:
            handler(_)
            after = len(state.get('last_messages', []))
            if after <= before + 1:
                push_message(success_message or f'{action_label} completed successfully.')
        except Exception as exc:
            push_message(f'{action_label} failed: {exc}')
            raise

    button.on_click(_wrapped)


run_selector.observe(on_run_selected, names='value')
scale_selector.observe(on_scale_selected, names='value')
model_selector.observe(on_model_selected, names='value')
bind_button_with_feedback(refresh_models_btn, lambda _: (refresh_model_options(), update_2d_plot(), update_rho_plots()), 'Refreshing model list', 'Model list refreshed.')
bind_button_with_feedback(export_segy_btn, on_export_selected_segy, 'Exporting selected model to SEGY')
x_slice_input.observe(lambda _: update_rho_plots(), names='value')
z_slice_input.observe(lambda _: update_rho_plots(), names='value')
bind_button_with_feedback(load_real_btn, on_load_real, 'Loading real data')
bind_button_with_feedback(load_syn_from_run_btn, on_load_syn_from_run, 'Loading synthetics from run')
bind_button_with_feedback(generate_syn_btn, on_generate_synthetics, 'Starting synthetic modelling')
bind_button_with_feedback(refresh_fd_btn, on_refresh_fd_status, 'Refreshing FD status')
bind_button_with_feedback(quit_btn, on_quit_gui, 'Shutting down results GUI server')

for w in [metric_select, tx_select, rx_local_select, trace_idx]:
    w.observe(update_data_compare_plot, names='value')

# Bind the view selector now that every handler it calls exists.
view_combo.observe(lambda _c: on_dataset_change(), names='value')
refresh_scale_options()
refresh_view_combos()

status_out.value = '\n'.join(state['last_messages'])

model_section = ipw.VBox([
    ipw.HTML('<h3>1) Models</h3>'),
    ipw.HBox([model_selector, refresh_models_btn, export_segy_btn]),
    export_status,
    model_plot_out,
    ipw.HTML('<b>Resistivity slices</b> (x for depth plot, z for x plot; dotted curves are the other scales of this ladder)'),
    ipw.HBox([x_slice_input, z_slice_input]),
    rho_z_out,
    rho_x_out,
])

data_section = ipw.VBox([
    ipw.HTML('<h3>2) Synthetics versus observed</h3>'),
    ipw.HBox([view_combo]),
    dataset_info,
    ipw.HBox([metric_select, tx_select]),
    ipw.HBox([rx_local_select, trace_idx]),
    data_compare_out,
    ipw.HBox([load_real_btn, load_syn_from_run_btn, nproc_input, generate_syn_btn, refresh_fd_btn]),
    fd_status,
])

tabs = ipw.Tab(children=[model_section, data_section])
tabs.set_title(0, 'Models')
tabs.set_title(1, 'Data')

layout = ipw.VBox([
    ipw.HTML('<h2>04_2d_inversion_results</h2>'),
    ipw.HBox([quit_btn], layout=ipw.Layout(justify_content='flex-end')),
    ipw.HBox([run_selector, scale_selector]),
    tabs,
    status_out,
])
display(layout)

state['initial_load'] = False
if run_selector.value:
    refresh_scale_options()
    refresh_model_options()

# Keep startup lightweight to avoid Voila init hangs.
with model_plot_out:
    clear_output(wait=True)
    display(ipw.HTML('Select a model to render the model comparison plot.'))
with rho_z_out:
    clear_output(wait=True)
with rho_x_out:
    clear_output(wait=True)
